In [1]:
import pandas as pd

# 1) 데이터 로드
df = pd.read_csv("dataset/merged_dataset.csv")  # 파일 이름은 네가 쓰는 걸로

# 2) 고독사율(death_rate) 계산
df = df.copy()
df = df[df["elderly_population"] > 0]  # 0으로 나누는 경우 제거
df["death_rate"] = df["target_value"] / df["elderly_population"]

# 3) (선택) 전체 인구 대비 노인 비율 추가
df["elderly_population_ratio"] = df["age_65_over"] / df["population_total"] * 100


In [2]:
feature_cols = [
    "single_household_ratio",              # 1인가구 비율
    "low_income_elderly_65_79_ratio",      # 저소득 65~79
    "low_income_elderly_80_over_ratio",    # 저소득 80+
    "aging_index",                         # 노령화지수
    "elderly_population_ratio",            # 65+ 인구 비율
]


In [3]:
# 4) death_rate 포함해서 상관계수 계산
corr = df[feature_cols + ["death_rate"]].corr()

# death_rate와의 상관계수 (절댓값)
corr_with_target = corr["death_rate"].loc[feature_cols].abs()

# 5) 합이 1이 되도록 정규화 → 전역 가중치
global_weights = corr_with_target / corr_with_target.sum()

print(global_weights)


single_household_ratio              0.368467
low_income_elderly_65_79_ratio      0.038977
low_income_elderly_80_over_ratio    0.095469
aging_index                         0.311710
elderly_population_ratio            0.185376
Name: death_rate, dtype: float64


In [4]:
region_weights_rows = []

for region_name, df_region in df.groupby("region"):
    # 연도가 너무 적거나 death_rate가 전부 같아서 상관계수 못 구하면 global 그대로 사용
    if df_region["death_rate"].nunique() <= 1 or len(df_region) < 3:
        for feat in feature_cols:
            region_weights_rows.append({
                "region": region_name,
                "feature": feat,
                "weight": global_weights[feat]
            })
        continue

    # 1) 이 구의 상관계수 행렬
    corr_r = df_region[feature_cols + ["death_rate"]].corr()
    corr_with_target_r = corr_r["death_rate"].loc[feature_cols].abs()

    # NaN(상수 피처 등) → 0으로 처리
    corr_with_target_r = corr_with_target_r.fillna(0.0)

    # 2) 중요도 순위
    sorted_feats = corr_with_target_r.sort_values(ascending=False).index.tolist()

    top_k = 2   # 가장 중요한 피처 개수
    bottom_k = 1  # 가장 덜 중요한 피처 개수

    top_feats = set(sorted_feats[:top_k])
    bottom_feats = set(sorted_feats[-bottom_k:]) if len(sorted_feats) >= bottom_k else set()

    # 3) 전역 가중치에 곱할 multiplier 지정
    adjusted = {}

    for feat in feature_cols:
        base_w = global_weights[feat]

        if feat in top_feats:
            m = 1.2
        elif feat in bottom_feats:
            m = 0.8
        else:
            m = 1.0

        adjusted[feat] = base_w * m

    # 4) 합이 1이 되도록 다시 정규화
    s = sum(adjusted.values())
    if s == 0:
        # 혹시 전부 0이면 (이론상 거의 없지만) 전역 가중치 그대로 사용
        for feat in feature_cols:
            region_weights_rows.append({
                "region": region_name,
                "feature": feat,
                "weight": global_weights[feat]
            })
    else:
        for feat in feature_cols:
            region_weights_rows.append({
                "region": region_name,
                "feature": feat,
                "weight": adjusted[feat] / s
            })

# 5) 최종 DataFrame → CSV로 저장
region_weights = pd.DataFrame(region_weights_rows)
region_weights.to_csv("region_feature_weights.csv", index=False, encoding="utf-8-sig")

region_weights.head()


,region,feature,weight
0,강남구,single_household_ratio,0.372259
1,강남구,low_income_elderly_65_79_ratio,0.047254
2,강남구,low_income_elderly_80_over_ratio,0.115742
3,강남구,aging_index,0.314918
4,강남구,elderly_population_ratio,0.149827
